# 구글 마운트

In [ ]:
from google.colab import drive

# 1. 구글 드라이브 연결
drive.mount('/content/drive')

Mounted at /content/drive


# 기상데이터

## 2024년

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/기상데이터_2024 (6월 1일부터).csv'

# encoding='cp949' 옵션을 추가해서 불러옵니다.
try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
    df = pd.read_csv(file_path, encoding='euc-kr')

missing_counts = df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
지점               0
지점명              0
일시               0
기온(°C)           1
강수량(mm)      17536
풍속(m/s)         32
습도(%)            1
일사(MJ/m2)    12084
전운량(10분위)        0
dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20544 entries, 0 to 20543
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   지점         20544 non-null  int64  
 1   지점명        20544 non-null  object 
 2   일시         20544 non-null  object 
 3   기온(°C)     20543 non-null  float64
 4   강수량(mm)    3008 non-null   float64
 5   풍속(m/s)    20512 non-null  float64
 6   습도(%)      20543 non-null  float64
 7   일사(MJ/m2)  8460 non-null   float64
 8   전운량(10분위)  20544 non-null  int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 1.4+ MB


In [ ]:
print(df.shape)

(20544, 9)


In [ ]:
# 강수량, 일사량 결측치 0으로 대체
df['강수량(mm)'] = df['강수량(mm)'].fillna(0)
df['일사(MJ/m2)'] = df['일사(MJ/m2)'].fillna(0)

# 기온, 습도, 풍속 결측치 보간법으로 대체
cols_to_avg = ['기온(°C)', '습도(%)', '풍속(m/s)']
df[cols_to_avg] = df[cols_to_avg].interpolate(method='linear', limit_direction='both')

# 결과 확인
print("--- 처리 후 결측치 개수 ---")
print(df.isnull().sum())

--- 처리 후 결측치 개수 ---
지점           0
지점명          0
일시           0
기온(°C)       0
강수량(mm)      0
풍속(m/s)      0
습도(%)        0
일사(MJ/m2)    0
전운량(10분위)    0
dtype: int64


In [ ]:
# 지점 번호(코드) 확인
print(df['지점'].unique())

# 지점 이름 확인
print(df['지점명'].unique())

[184 185 188 189]
['제주' '고산' '성산' '서귀포']


In [ ]:
print(df.columns.tolist())

['지점', '지점명', '일시', '기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '일사(MJ/m2)', '전운량(10분위)']


In [ ]:
# 시계열 데이터 형식으로 변환
df['일시'] = pd.to_datetime(df['일시'])

# 칼럼 만드는거
df['날짜'] = df['일시'].dt.date     # YYYY-MM-DD 형태
df['시간'] = df['일시'].dt.hour     # 0~23 숫자 형태 (분석하기 더 좋음)

# 만약 시간도 '00:00' 같은 문자열 형태를 원하신다면 아래 코드를 쓰세요.
df['시간'] = df['일시'].dt.time

# 맨 앞에 나오게
cols = ['날짜', '시간'] + [col for col in df.columns if col not in ['날짜', '시간', '일시']]
df = df[cols]

display(df.head())
display(df.shape)

,날짜,시간,지점,지점명,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2024-06-01,00:00:00,184,제주,18.4,0.0,2.1,78.0,0.0,6
1,2024-06-01,01:00:00,184,제주,18.1,0.0,2.6,81.0,0.0,7
2,2024-06-01,02:00:00,184,제주,17.8,0.0,2.2,77.0,0.0,6
3,2024-06-01,03:00:00,184,제주,17.8,0.0,2.4,74.0,0.0,5
4,2024-06-01,04:00:00,184,제주,18.0,0.0,2.2,77.0,0.0,7


(20544, 10)

In [ ]:
print(df.columns.tolist())

['날짜', '시간', '지점', '지점명', '기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '일사(MJ/m2)', '전운량(10분위)']


In [ ]:
import pandas as pd

# 1. 수치 데이터가 있는 칼럼들만 선택 (지점, 지점명 제외)
# 전운량도 수치 데이터이므로 포함했습니다.
target_cols = ['기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '일사(MJ/m2)', '전운량(10분위)']

# 2. 날짜와 시간을 기준으로 그룹화하여 평균 계산
# reset_index()를 해줘야 '날짜'와 '시간'이 다시 일반 칼럼으로 돌아옵니다.
df_final = df.groupby(['날짜', '시간'])[target_cols].mean().reset_index()

df_final['전운량(10분위)'] = df_final['전운량(10분위)'].round(0).astype(int)

# 3. 결과 확인
print(f"통합 전 크기: {df.shape}")
print(f"통합 후 크기: {df_final.shape}") # 행 수가 4분의 1 근처로 줄어듭니다.

display(df_final.head(10))

통합 전 크기: (20544, 10)
통합 후 크기: (5136, 8)


,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2024-06-01,00:00:00,17.750,0.0,2.775,80.50,0.0000,3
1,2024-06-01,01:00:00,17.775,0.0,3.675,79.75,0.0000,8
2,2024-06-01,02:00:00,17.625,0.0,2.650,78.50,0.0000,6
3,2024-06-01,03:00:00,17.650,0.0,3.025,78.00,0.0000,5
4,2024-06-01,04:00:00,17.600,0.0,3.175,80.00,0.0000,7
5,2024-06-01,05:00:00,17.600,0.0,3.425,84.50,0.0000,8
6,2024-06-01,06:00:00,17.925,0.0,2.500,80.25,0.0100,8
7,2024-06-01,07:00:00,18.825,0.0,3.225,76.50,0.2025,8
8,2024-06-01,08:00:00,19.775,0.0,3.200,76.50,0.5300,9
9,2024-06-01,09:00:00,20.300,0.0,3.900,77.25,0.4900,8


In [ ]:
# 2024년 기상데이터 전처리한거
df_weather_24 = df_final.copy()

display(df_weather_24.head(10))
print(df_weather_24.shape)

,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2024-06-01,00:00:00,17.750,0.0,2.775,80.50,0.0000,3
1,2024-06-01,01:00:00,17.775,0.0,3.675,79.75,0.0000,8
2,2024-06-01,02:00:00,17.625,0.0,2.650,78.50,0.0000,6
3,2024-06-01,03:00:00,17.650,0.0,3.025,78.00,0.0000,5
4,2024-06-01,04:00:00,17.600,0.0,3.175,80.00,0.0000,7
5,2024-06-01,05:00:00,17.600,0.0,3.425,84.50,0.0000,8
6,2024-06-01,06:00:00,17.925,0.0,2.500,80.25,0.0100,8
7,2024-06-01,07:00:00,18.825,0.0,3.225,76.50,0.2025,8
8,2024-06-01,08:00:00,19.775,0.0,3.200,76.50,0.5300,9
9,2024-06-01,09:00:00,20.300,0.0,3.900,77.25,0.4900,8


(5136, 8)


## 2025년

In [ ]:
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/기상데이터_2025.csv'

# encoding='cp949' 옵션을 추가해서 불러옵니다.
try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
    df = pd.read_csv(file_path, encoding='euc-kr')

missing_counts = df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
지점               0
지점명              0
일시               0
기온(°C)          18
강수량(mm)      30991
풍속(m/s)         28
습도(%)           18
일사(MJ/m2)    20639
전운량(10분위)        3
dtype: int64


In [ ]:
print(df.shape)

(35031, 9)


In [ ]:
# 지점 번호(코드) 확인
print(df['지점'].unique())

# 지점 이름 확인
print(df['지점명'].unique())

[184 185 188 189]
['제주' '고산' '성산' '서귀포']


In [ ]:
# 강수량, 일사량 결측치 0으로 대체
df['강수량(mm)'] = df['강수량(mm)'].fillna(0)
df['일사(MJ/m2)'] = df['일사(MJ/m2)'].fillna(0)

# 기온, 습도, 풍속 결측치 보간법으로 대체
cols_to_avg = ['기온(°C)', '습도(%)', '풍속(m/s)', '전운량(10분위)']
df[cols_to_avg] = df[cols_to_avg].interpolate(method='linear', limit_direction='both')


# 결과 확인
print("--- 처리 후 결측치 개수 ---")
print(df.isnull().sum())

--- 처리 후 결측치 개수 ---
지점           0
지점명          0
일시           0
기온(°C)       0
강수량(mm)      0
풍속(m/s)      0
습도(%)        0
일사(MJ/m2)    0
전운량(10분위)    0
dtype: int64


In [ ]:
# 시계열 데이터 형식으로 변환
df['일시'] = pd.to_datetime(df['일시'])

# 칼럼 만드는거
df['날짜'] = df['일시'].dt.date     # YYYY-MM-DD 형태
df['시간'] = df['일시'].dt.hour     # 0~23 숫자 형태 (분석하기 더 좋음)

# 만약 시간도 '00:00' 같은 문자열 형태를 원하신다면 아래 코드를 쓰세요.
df['시간'] = df['일시'].dt.time

# 맨 앞에 나오게
cols = ['날짜', '시간'] + [col for col in df.columns if col not in ['날짜', '시간', '일시']]
df = df[cols]

display(df.head())
display(df.shape)

,날짜,시간,지점,지점명,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2025-01-01,00:00:00,184,제주,6.1,0.0,2.7,57.0,0.0,8.0
1,2025-01-01,01:00:00,184,제주,6.1,0.0,3.0,59.0,0.0,8.0
2,2025-01-01,02:00:00,184,제주,6.1,0.0,2.0,59.0,0.0,8.0
3,2025-01-01,03:00:00,184,제주,6.0,0.0,2.4,60.0,0.0,9.0
4,2025-01-01,04:00:00,184,제주,5.6,0.0,2.1,59.0,0.0,8.0


(35031, 10)

In [ ]:
import pandas as pd

# 1. 수치 데이터가 있는 칼럼들만 선택 (지점, 지점명 제외)
# 전운량도 수치 데이터이므로 포함했습니다.
target_cols = ['기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '일사(MJ/m2)', '전운량(10분위)']

# 2. 날짜와 시간을 기준으로 그룹화하여 평균 계산
# reset_index()를 해줘야 '날짜'와 '시간'이 다시 일반 칼럼으로 돌아옵니다.
df_final = df.groupby(['날짜', '시간'])[target_cols].mean().reset_index()

df_final['전운량(10분위)'] = df_final['전운량(10분위)'].round(0).astype(int)

# 3. 결과 확인
print(f"통합 전 크기: {df.shape}")
print(f"통합 후 크기: {df_final.shape}") # 행 수가 4분의 1 근처로 줄어듭니다.

display(df_final.head(10))

통합 전 크기: (35031, 10)
통합 후 크기: (8760, 8)


,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2025-01-01,00:00:00,5.425,0.0,3.975,64.25,0.0000,2
1,2025-01-01,01:00:00,5.300,0.0,3.625,64.00,0.0000,2
2,2025-01-01,02:00:00,5.225,0.0,3.425,64.00,0.0000,2
3,2025-01-01,03:00:00,5.275,0.0,3.550,66.00,0.0000,4
4,2025-01-01,04:00:00,5.150,0.0,3.100,67.00,0.0000,4
5,2025-01-01,05:00:00,5.050,0.0,2.625,67.50,0.0000,4
6,2025-01-01,06:00:00,5.075,0.0,3.700,68.50,0.0000,3
7,2025-01-01,07:00:00,5.075,0.0,3.150,67.75,0.0000,3
8,2025-01-01,08:00:00,5.275,0.0,3.050,66.75,0.0075,3
9,2025-01-01,09:00:00,6.875,0.0,1.800,61.75,0.2400,4


In [ ]:
# 2025년 기상데이터 전처리한거
df_weather_25 = df_final.copy()

display(df_weather_25.head(10))
print(df_weather_25.shape)

,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2025-01-01,00:00:00,5.425,0.0,3.975,64.25,0.0000,2
1,2025-01-01,01:00:00,5.300,0.0,3.625,64.00,0.0000,2
2,2025-01-01,02:00:00,5.225,0.0,3.425,64.00,0.0000,2
3,2025-01-01,03:00:00,5.275,0.0,3.550,66.00,0.0000,4
4,2025-01-01,04:00:00,5.150,0.0,3.100,67.00,0.0000,4
5,2025-01-01,05:00:00,5.050,0.0,2.625,67.50,0.0000,4
6,2025-01-01,06:00:00,5.075,0.0,3.700,68.50,0.0000,3
7,2025-01-01,07:00:00,5.075,0.0,3.150,67.75,0.0000,3
8,2025-01-01,08:00:00,5.275,0.0,3.050,66.75,0.0075,3
9,2025-01-01,09:00:00,6.875,0.0,1.800,61.75,0.2400,4


(8760, 8)


## 2026년

In [ ]:
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/기상데이터_2026 (5월 7일까지).csv'

# encoding='cp949' 옵션을 추가해서 불러옵니다.
try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
    df = pd.read_csv(file_path, encoding='euc-kr')

missing_counts = df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
지점               0
지점명              0
일시               0
기온(°C)           0
강수량(mm)      10821
풍속(m/s)          0
습도(%)            0
일사(MJ/m2)     7338
전운량(10분위)        0
dtype: int64


In [ ]:
print(df.shape)

(12136, 9)


In [ ]:
# 지점 번호(코드) 확인
print(df['지점'].unique())

# 지점 이름 확인
print(df['지점명'].unique())

[184 185 188 189]
['제주' '고산' '성산' '서귀포']


In [ ]:
# 강수량, 일사량 결측치 0으로 대체
df['강수량(mm)'] = df['강수량(mm)'].fillna(0)
df['일사(MJ/m2)'] = df['일사(MJ/m2)'].fillna(0)

# 결과 확인
print("--- 처리 후 결측치 개수 ---")
print(df.isnull().sum())

--- 처리 후 결측치 개수 ---
지점           0
지점명          0
일시           0
기온(°C)       0
강수량(mm)      0
풍속(m/s)      0
습도(%)        0
일사(MJ/m2)    0
전운량(10분위)    0
dtype: int64


In [ ]:
# 시계열 데이터 형식으로 변환
df['일시'] = pd.to_datetime(df['일시'])

# 칼럼 만드는거
df['날짜'] = df['일시'].dt.date     # YYYY-MM-DD 형태
df['시간'] = df['일시'].dt.hour     # 0~23 숫자 형태 (분석하기 더 좋음)

# 만약 시간도 '00:00' 같은 문자열 형태를 원하신다면 아래 코드를 쓰세요.
df['시간'] = df['일시'].dt.time

# 맨 앞에 나오게
cols = ['날짜', '시간'] + [col for col in df.columns if col not in ['날짜', '시간', '일시']]
df = df[cols]

display(df.head())
display(df.shape)

,날짜,시간,지점,지점명,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2026-01-01,00:00:00,184,제주,1.9,0.0,6.0,58,0.0,6
1,2026-01-01,01:00:00,184,제주,1.7,0.0,5.9,55,0.0,6
2,2026-01-01,02:00:00,184,제주,1.8,0.0,5.3,62,0.0,6
3,2026-01-01,03:00:00,184,제주,1.7,0.0,5.7,59,0.0,6
4,2026-01-01,04:00:00,184,제주,1.7,0.0,5.6,57,0.0,7


(12136, 10)

In [ ]:
import pandas as pd

# 1. 수치 데이터가 있는 칼럼들만 선택 (지점, 지점명 제외)
# 전운량도 수치 데이터이므로 포함했습니다.
target_cols = ['기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '일사(MJ/m2)', '전운량(10분위)']

# 2. 날짜와 시간을 기준으로 그룹화하여 평균 계산
# reset_index()를 해줘야 '날짜'와 '시간'이 다시 일반 칼럼으로 돌아옵니다.
df_final = df.groupby(['날짜', '시간'])[target_cols].mean().reset_index()

df_final['전운량(10분위)'] = df_final['전운량(10분위)'].round(0).astype(int)

# 3. 결과 확인
print(f"통합 전 크기: {df.shape}")
print(f"통합 후 크기: {df_final.shape}") # 행 수가 4분의 1 근처로 줄어듭니다.

display(df_final.head(10))

통합 전 크기: (12136, 10)
통합 후 크기: (3034, 8)


,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2026-01-01,00:00:00,2.750,0.0,5.975,63.50,0.0000,5
1,2026-01-01,01:00:00,2.300,0.0,4.775,61.75,0.0000,5
2,2026-01-01,02:00:00,2.375,0.0,4.800,65.00,0.0000,5
3,2026-01-01,03:00:00,2.150,0.0,3.900,64.75,0.0000,4
4,2026-01-01,04:00:00,2.125,0.0,4.250,63.50,0.0000,4
5,2026-01-01,05:00:00,1.950,0.0,4.175,63.00,0.0000,4
6,2026-01-01,06:00:00,2.200,0.0,4.275,64.00,0.0000,6
7,2026-01-01,07:00:00,2.050,0.0,4.600,63.75,0.0000,6
8,2026-01-01,08:00:00,2.050,0.0,4.975,63.75,0.0075,8
9,2026-01-01,09:00:00,2.475,0.0,4.650,63.75,0.1050,8


In [ ]:
# 2026년 기상데이터 전처리한거
df_weather_26 = df_final.copy()

display(df_weather_26.head(10))
print(df_weather_26.shape)

,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2026-01-01,00:00:00,2.750,0.0,5.975,63.50,0.0000,5
1,2026-01-01,01:00:00,2.300,0.0,4.775,61.75,0.0000,5
2,2026-01-01,02:00:00,2.375,0.0,4.800,65.00,0.0000,5
3,2026-01-01,03:00:00,2.150,0.0,3.900,64.75,0.0000,4
4,2026-01-01,04:00:00,2.125,0.0,4.250,63.50,0.0000,4
5,2026-01-01,05:00:00,1.950,0.0,4.175,63.00,0.0000,4
6,2026-01-01,06:00:00,2.200,0.0,4.275,64.00,0.0000,6
7,2026-01-01,07:00:00,2.050,0.0,4.600,63.75,0.0000,6
8,2026-01-01,08:00:00,2.050,0.0,4.975,63.75,0.0075,8
9,2026-01-01,09:00:00,2.475,0.0,4.650,63.75,0.1050,8


(3034, 8)


In [ ]:
df_weather_26.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3034 entries, 0 to 3033
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   날짜         3034 non-null   object 
 1   시간         3034 non-null   object 
 2   기온(°C)     3034 non-null   float64
 3   강수량(mm)    3034 non-null   float64
 4   풍속(m/s)    3034 non-null   float64
 5   습도(%)      3034 non-null   float64
 6   일사(MJ/m2)  3034 non-null   float64
 7   전운량(10분위)  3034 non-null   int64  
dtypes: float64(5), int64(1), object(2)
memory usage: 189.8+ KB


## 24-26년 기상 데이터 합치는거

In [ ]:
import pandas as pd

# 1. 리스트에 순서대로 담아서 위아래(axis=0)로 합치기
df_total_weather = pd.concat([df_weather_24, df_weather_25, df_weather_26], axis=0)

# 2. 합치고 나면 인덱스가 0,1,2...0,1,2... 식으로 중복될 수 있으니 초기화해줍니다.
df_total_weather = df_total_weather.reset_index(drop=True)

# 3. 날짜와 시간 순서대로 완벽하게 정렬 (혹시 모를 순서 꼬임 방지)
df_total_weather = df_total_weather.sort_values(by=['날짜', '시간']).reset_index(drop=True)

# 4. 결과 확인
display(df_total_weather.head())
print(f"24~26년 통합 데이터 크기: {df_total_weather.shape}")

,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
0,2024-06-01,00:00:00,17.750,0.0,2.775,80.50,0.0,3
1,2024-06-01,01:00:00,17.775,0.0,3.675,79.75,0.0,8
2,2024-06-01,02:00:00,17.625,0.0,2.650,78.50,0.0,6
3,2024-06-01,03:00:00,17.650,0.0,3.025,78.00,0.0,5
4,2024-06-01,04:00:00,17.600,0.0,3.175,80.00,0.0,7


24~26년 통합 데이터 크기: (16930, 8)


In [ ]:
display(df_total_weather.tail())

,날짜,시간,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위)
16925,2026-05-07,05:00:00,14.950,0.0,2.175,78.75,0.0000,5
16926,2026-05-07,06:00:00,14.800,0.0,2.325,77.75,0.0025,4
16927,2026-05-07,07:00:00,16.225,0.0,1.975,73.25,0.2675,6
16928,2026-05-07,08:00:00,18.725,0.0,1.500,68.50,0.6875,6
16929,2026-05-07,09:00:00,20.200,0.0,2.875,64.75,1.2050,7


In [ ]:
# 저장할 파일 경로 설정 (기존 폴더 경로와 맞춰두었습니다)
output_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/통합_기상데이터_24_26.csv'

# 데이터프레임을 CSV 파일로 저장
# 한글 깨짐 방지를 위해 encoding='cp949'를 사용합니다.
df_total_weather.to_csv(output_path, index=False, encoding='cp949')

print(f"✅ 저장 완료! 경로: {output_path}")

✅ 저장 완료! 경로: /content/drive/MyDrive/ax_team/ax_data/model_a/통합_기상데이터_24_26.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/통합_기상데이터_24_26.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


# 일 정보

In [ ]:
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/일_정보.csv'

# encoding='cp949' 옵션을 추가해서 불러옵니다.
try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
    df = pd.read_csv(file_path, encoding='euc-kr')

df.columns = ['날짜', '요일', '휴일_유무', '휴일_종류']

missing_counts = df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
날짜       0
요일       0
휴일_유무    0
휴일_종류    0
dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 706 entries, 0 to 705
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   날짜      706 non-null    object
 1   요일      706 non-null    int64 
 2   휴일_유무   706 non-null    int64 
 3   휴일_종류   706 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 22.2+ KB


In [ ]:
print(df.shape)

(706, 4)


In [ ]:
display(df.head())

,날짜,요일,휴일_유무,휴일_종류
0,2024-06-01,5,1,0
1,2024-06-02,6,1,0
2,2024-06-03,0,0,0
3,2024-06-04,1,0,0
4,2024-06-05,2,0,0


In [ ]:
import pandas as pd
import numpy as np

# 1. '일_정보' 데이터 확장 (24배 늘리기)
df_expanded = df.loc[df.index.repeat(24)].reset_index(drop=True).copy()

# 2. 시간 칼럼을 '00:00:00' 형식의 문자열로 생성
# 0~23 숫자를 만들고, 뒤에 ':00:00'을 붙여서 00:00:00 형식을 만듭니다.
hours = np.tile(np.arange(24), len(df))
df_expanded['시간'] = [f"{h:02d}:00:00" for h in hours]

# 3. 날짜 형식 깔끔하게 정리 (문자열 또는 date 객체)
df_expanded['날짜'] = pd.to_datetime(df_expanded['날짜']).dt.strftime('%Y-%m-%d')

# 4. 필요한 칼럼만 순서대로 보기 좋게 배치
# 원래 있던 'date' 칼럼은 '날짜'로 대체했으니 빼줍니다.
cols = ['날짜', '시간', '요일', '휴일_유무', '휴일_종류']
df_expanded = df_expanded[cols]

# 5. 결과 확인
display(df_expanded.head(25))

,날짜,시간,요일,휴일_유무,휴일_종류
0,2024-06-01,00:00:00,5,1,0
1,2024-06-01,01:00:00,5,1,0
2,2024-06-01,02:00:00,5,1,0
3,2024-06-01,03:00:00,5,1,0
4,2024-06-01,04:00:00,5,1,0
5,2024-06-01,05:00:00,5,1,0
6,2024-06-01,06:00:00,5,1,0
7,2024-06-01,07:00:00,5,1,0
8,2024-06-01,08:00:00,5,1,0
9,2024-06-01,09:00:00,5,1,0


In [ ]:
df_expanded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16944 entries, 0 to 16943
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   날짜      16944 non-null  object
 1   시간      16944 non-null  object
 2   요일      16944 non-null  int64 
 3   휴일_유무   16944 non-null  int64 
 4   휴일_종류   16944 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 662.0+ KB


In [ ]:
import pandas as pd
import numpy as np

# 1. 시간 문자열('00:00:00')에서 숫자 시간만 추출 (예: 0, 1, 2...)
# 위에서 만든 df_expanded['시간']을 활용합니다.
hour_numeric = np.tile(np.arange(24), len(df))

# 2. Sin, Cos 변환값 계산
# 시간은 24시간 주기이므로 2 * pi * (현재시간 / 전체시간) 공식을 사용합니다.
df_expanded['hour_sin'] = np.sin(2 * np.pi * hour_numeric / 24)
df_expanded['hour_cos'] = np.cos(2 * np.pi * hour_numeric / 24)

# 3. 결과 확인 (순서 조정)
cols = ['날짜', '시간', 'hour_sin','hour_cos','요일', '휴일_유무', '휴일_종류']
df_expanded = df_expanded[cols]

display(df_expanded.head(25))

,날짜,시간,hour_sin,hour_cos,요일,휴일_유무,휴일_종류
0,2024-06-01,00:00:00,0.000000e+00,1.000000e+00,5,1,0
1,2024-06-01,01:00:00,2.588190e-01,9.659258e-01,5,1,0
2,2024-06-01,02:00:00,5.000000e-01,8.660254e-01,5,1,0
3,2024-06-01,03:00:00,7.071068e-01,7.071068e-01,5,1,0
4,2024-06-01,04:00:00,8.660254e-01,5.000000e-01,5,1,0
5,2024-06-01,05:00:00,9.659258e-01,2.588190e-01,5,1,0
6,2024-06-01,06:00:00,1.000000e+00,6.123234e-17,5,1,0
7,2024-06-01,07:00:00,9.659258e-01,-2.588190e-01,5,1,0
8,2024-06-01,08:00:00,8.660254e-01,-5.000000e-01,5,1,0
9,2024-06-01,09:00:00,7.071068e-01,-7.071068e-01,5,1,0


In [ ]:
df_expanded['주말_유무'] = df_expanded['요일'].isin([5, 6]).astype(int)

cols = ['날짜', '시간', 'hour_sin','hour_cos','요일','주말_유무', '휴일_유무', '휴일_종류']
df_expanded = df_expanded[cols]

display(df_expanded.head(25))

,날짜,시간,hour_sin,hour_cos,요일,주말_유무,휴일_유무,휴일_종류
0,2024-06-01,00:00:00,0.000000e+00,1.000000e+00,5,1,1,0
1,2024-06-01,01:00:00,2.588190e-01,9.659258e-01,5,1,1,0
2,2024-06-01,02:00:00,5.000000e-01,8.660254e-01,5,1,1,0
3,2024-06-01,03:00:00,7.071068e-01,7.071068e-01,5,1,1,0
4,2024-06-01,04:00:00,8.660254e-01,5.000000e-01,5,1,1,0
5,2024-06-01,05:00:00,9.659258e-01,2.588190e-01,5,1,1,0
6,2024-06-01,06:00:00,1.000000e+00,6.123234e-17,5,1,1,0
7,2024-06-01,07:00:00,9.659258e-01,-2.588190e-01,5,1,1,0
8,2024-06-01,08:00:00,8.660254e-01,-5.000000e-01,5,1,1,0
9,2024-06-01,09:00:00,7.071068e-01,-7.071068e-01,5,1,1,0


In [ ]:
day_info = df_expanded.copy()

print(day_info.shape)
display(day_info.head(25))

(16944, 8)


,날짜,시간,hour_sin,hour_cos,요일,주말_유무,휴일_유무,휴일_종류
0,2024-06-01,00:00:00,0.000000e+00,1.000000e+00,5,1,1,0
1,2024-06-01,01:00:00,2.588190e-01,9.659258e-01,5,1,1,0
2,2024-06-01,02:00:00,5.000000e-01,8.660254e-01,5,1,1,0
3,2024-06-01,03:00:00,7.071068e-01,7.071068e-01,5,1,1,0
4,2024-06-01,04:00:00,8.660254e-01,5.000000e-01,5,1,1,0
5,2024-06-01,05:00:00,9.659258e-01,2.588190e-01,5,1,1,0
6,2024-06-01,06:00:00,1.000000e+00,6.123234e-17,5,1,1,0
7,2024-06-01,07:00:00,9.659258e-01,-2.588190e-01,5,1,1,0
8,2024-06-01,08:00:00,8.660254e-01,-5.000000e-01,5,1,1,0
9,2024-06-01,09:00:00,7.071068e-01,-7.071068e-01,5,1,1,0


In [ ]:
display(day_info.tail(25))

,날짜,시간,hour_sin,hour_cos,요일,주말_유무,휴일_유무,휴일_종류
16919,2026-05-06,23:00:00,-2.588190e-01,9.659258e-01,2,0,0,0
16920,2026-05-07,00:00:00,0.000000e+00,1.000000e+00,3,0,0,0
16921,2026-05-07,01:00:00,2.588190e-01,9.659258e-01,3,0,0,0
16922,2026-05-07,02:00:00,5.000000e-01,8.660254e-01,3,0,0,0
16923,2026-05-07,03:00:00,7.071068e-01,7.071068e-01,3,0,0,0
16924,2026-05-07,04:00:00,8.660254e-01,5.000000e-01,3,0,0,0
16925,2026-05-07,05:00:00,9.659258e-01,2.588190e-01,3,0,0,0
16926,2026-05-07,06:00:00,1.000000e+00,6.123234e-17,3,0,0,0
16927,2026-05-07,07:00:00,9.659258e-01,-2.588190e-01,3,0,0,0
16928,2026-05-07,08:00:00,8.660254e-01,-5.000000e-01,3,0,0,0


In [ ]:
# 저장할 파일 경로 설정 (기존 폴더 경로와 맞춰두었습니다)
output_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/day_info.csv'

# 데이터프레임을 CSV 파일로 저장
# 한글 깨짐 방지를 위해 encoding='cp949'를 사용합니다.
day_info.to_csv(output_path, index=False, encoding='cp949')

print(f"✅ 저장 완료! 경로: {output_path}")

✅ 저장 완료! 경로: /content/drive/MyDrive/ax_team/ax_data/model_a/day_info.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/day_info.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


# 평년 기온 대비 편차

## 00-09 까지 평균기온

In [ ]:
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/평기대편1.csv'

# encoding='cp949' 옵션을 추가해서 불러옵니다.
try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
    df = pd.read_csv(file_path, encoding='euc-kr')

missing_counts = df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
지점          0
지점명         0
일시          0
평균기온(°C)    0
dtype: int64


In [ ]:
branch_counts = df['지점명'].value_counts()

print("--- 각 지점명별 데이터 개수 ---")
print(branch_counts)

--- 각 지점명별 데이터 개수 ---
지점명
성산     6821
제주     3653
고산     3653
서귀포    3653
성산포    2647
Name: count, dtype: int64


In [ ]:
# 1. 지점명이 '성산'이면서 지점 번호가 187인 데이터 제외
# 2. 지점명이 '성산포'인 데이터 제외
df_filtered = df[~((df['지점명'] == '성산') & (df['지점'] == 187)) &
                 (df['지점명'] != '성산포')].copy()

# 삭제 결과 확인
print(f"삭제 전 데이터 개수: {len(df)}")
print(f"삭제 후 데이터 개수: {len(df_filtered)}")
print("\n--- 남은 지점명 확인 ---")
print(df_filtered['지점명'].value_counts())

삭제 전 데이터 개수: 20427
삭제 후 데이터 개수: 14612

--- 남은 지점명 확인 ---
지점명
제주     3653
고산     3653
성산     3653
서귀포    3653
Name: count, dtype: int64


In [ ]:
import pandas as pd

# 2. 지점명별로 그룹화하여 시작일(min)과 종료일(max) 계산
time_range = df_filtered.groupby('지점명')['일시'].agg(['min', 'max'])

# 3. 칼럼 이름 보기 좋게 변경
time_range.columns = ['시작일시', '종료일시']

print("--- 각 지점별 데이터 기간 ---")
print(time_range)

--- 각 지점별 데이터 기간 ---
           시작일시        종료일시
지점명                        
고산   2000-01-01  2009-12-31
서귀포  2000-01-01  2009-12-31
성산   2000-01-01  2009-12-31
제주   2000-01-01  2009-12-31


In [ ]:
# 1. 먼저 '일시' 칼럼이 날짜 형식인지 확인하고 변환합니다.
df_filtered['일시'] = pd.to_datetime(df_filtered['일시'])

# 2. '일시'를 기준으로 그룹화한 뒤, 숫자 데이터들만 평균(mean)을 냅니다.
# '지점'이나 '지점명' 같은 문자열/ID 칼럼은 평균을 낼 수 없으므로 제외됩니다.
df_daily_avg = df_filtered.groupby('일시').mean(numeric_only=True).reset_index()

df_daily_avg = df_daily_avg.drop(columns=['지점'])

# 3. 결과 확인
print("--- 통합된 데이터 상위 5행 ---")
print(df_daily_avg.head())

print(f"\n통합 전 행 개수: {len(df_filtered)}")
print(f"통합 후 행 개수: {len(df_daily_avg)}")

--- 통합된 데이터 상위 5행 ---
          일시  평균기온(°C)
0 2000-01-01    12.775
1 2000-01-02    11.425
2 2000-01-03     7.375
3 2000-01-04    10.500
4 2000-01-05    14.525

통합 전 행 개수: 14612
통합 후 행 개수: 3653


In [ ]:
display(df_daily_avg.head())

,일시,평균기온(°C)
0,2000-01-01,12.775
1,2000-01-02,11.425
2,2000-01-03,7.375
3,2000-01-04,10.500
4,2000-01-05,14.525


In [ ]:
# 1. '일시'에서 월(month)과 일(day) 정보를 추출합니다.
df_daily_avg['월'] = df_daily_avg['일시'].dt.month
df_daily_avg['일'] = df_daily_avg['일시'].dt.day

# 2. 월, 일을 기준으로 그룹화하여 평균기온의 평균을 계산합니다.
# '평균기온(°C)' 칼럼명은 실제 데이터프레임의 칼럼명으로 맞춰주세요.
final_climate_data = df_daily_avg.groupby(['월', '일'])['평균기온(°C)'].mean().reset_index()

# 3. 결과 확인 (1월 1일부터 12월 31일까지 총 366일 데이터)
print("--- 10개년 월-일별 평균 기온 ---")
display(final_climate_data.head())
print(f"\n최종 행 개수: {len(final_climate_data)} (366일에 가까울수록 정상)")

--- 10개년 월-일별 평균 기온 ---


,월,일,평균기온(°C)
0,1,1,6.7700
1,1,2,7.1975
2,1,3,7.0625
3,1,4,6.5325
4,1,5,6.5400



최종 행 개수: 366 (366일에 가까울수록 정상)


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_00_09.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
final_climate_data.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_00_09.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_00_09.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


## 10-19 까지 평균기온

In [ ]:
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/평기대편2.csv'

# encoding='cp949' 옵션을 추가해서 불러옵니다.
try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
    df = pd.read_csv(file_path, encoding='euc-kr')

missing_counts = df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
지점          0
지점명         0
일시          0
평균기온(°C)    0
dtype: int64


In [ ]:
branch_counts = df['지점명'].value_counts()

print("--- 각 지점명별 데이터 개수 ---")
print(branch_counts)

--- 각 지점명별 데이터 개수 ---
지점명
제주     3652
성산     3652
서귀포    3651
고산     3649
Name: count, dtype: int64


In [ ]:
import pandas as pd


# 2. 지점명별로 그룹화하여 시작일(min)과 종료일(max) 계산
time_range = df.groupby('지점명')['일시'].agg(['min', 'max'])

# 3. 칼럼 이름 보기 좋게 변경
time_range.columns = ['시작일시', '종료일시']

print("--- 각 지점별 데이터 기간 ---")
display(time_range)

--- 각 지점별 데이터 기간 ---


,시작일시,종료일시
지점명,,
고산,2010-01-01,2019-12-31
서귀포,2010-01-01,2019-12-31
성산,2010-01-01,2019-12-31
제주,2010-01-01,2019-12-31


In [ ]:
# 1. 먼저 '일시' 칼럼이 날짜 형식인지 확인하고 변환합니다.
df['일시'] = pd.to_datetime(df['일시'])

# 2. '일시'를 기준으로 그룹화한 뒤, 숫자 데이터들만 평균(mean)을 냅니다.
# '지점'이나 '지점명' 같은 문자열/ID 칼럼은 평균을 낼 수 없으므로 제외됩니다.
df_daily_avg = df.groupby('일시').mean(numeric_only=True).reset_index()

df_daily_avg = df_daily_avg.drop(columns=['지점'])

# 3. 결과 확인
print("--- 통합된 데이터 상위 5행 ---")
print(df_daily_avg.head())

print(f"\n통합 전 행 개수: {len(df)}")
print(f"통합 후 행 개수: {len(df_daily_avg)}")

--- 통합된 데이터 상위 5행 ---
          일시  평균기온(°C)
0 2010-01-01     3.800
1 2010-01-02     9.450
2 2010-01-03     4.775
3 2010-01-04     7.700
4 2010-01-05     1.825

통합 전 행 개수: 14604
통합 후 행 개수: 3652


In [ ]:
# 1. '일시'에서 월(month)과 일(day) 정보를 추출합니다.
df_daily_avg['월'] = df_daily_avg['일시'].dt.month
df_daily_avg['일'] = df_daily_avg['일시'].dt.day

# 2. 월, 일을 기준으로 그룹화하여 평균기온의 평균을 계산합니다.
# '평균기온(°C)' 칼럼명은 실제 데이터프레임의 칼럼명으로 맞춰주세요.
final_climate_data = df_daily_avg.groupby(['월', '일'])['평균기온(°C)'].mean().reset_index()

# 3. 결과 확인 (1월 1일부터 12월 31일까지 총 366일 데이터)
print("--- 10개년 월-일별 평균 기온 ---")
display(final_climate_data.head())
print(f"\n최종 행 개수: {len(final_climate_data)} (366일에 가까울수록 정상)")

--- 10개년 월-일별 평균 기온 ---


,월,일,평균기온(°C)
0,1,1,6.0850
1,1,2,7.2800
2,1,3,6.5100
3,1,4,6.7600
4,1,5,6.5175



최종 행 개수: 366 (366일에 가까울수록 정상)


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_10_19.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
final_climate_data.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_10_19.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_10_19.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


## 20-26 까지 평균기온



In [ ]:
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/평기대편3.csv'

# encoding='cp949' 옵션을 추가해서 불러옵니다.
try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
    df = pd.read_csv(file_path, encoding='euc-kr')

missing_counts = df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
지점          0
지점명         0
일시          0
평균기온(°C)    0
dtype: int64


In [ ]:
branch_counts = df['지점명'].value_counts()

print("--- 각 지점명별 데이터 개수 ---")
print(branch_counts)

--- 각 지점명별 데이터 개수 ---
지점명
제주     2318
고산     2318
서귀포    2318
성산     2316
Name: count, dtype: int64


In [ ]:
import pandas as pd


# 2. 지점명별로 그룹화하여 시작일(min)과 종료일(max) 계산
time_range = df.groupby('지점명')['일시'].agg(['min', 'max'])

# 3. 칼럼 이름 보기 좋게 변경
time_range.columns = ['시작일시', '종료일시']

print("--- 각 지점별 데이터 기간 ---")
display(time_range)

--- 각 지점별 데이터 기간 ---


,시작일시,종료일시
지점명,,
고산,2020-01-01,2026-05-06
서귀포,2020-01-01,2026-05-06
성산,2020-01-01,2026-05-06
제주,2020-01-01,2026-05-06


In [ ]:
# 1. 먼저 '일시' 칼럼이 날짜 형식인지 확인하고 변환합니다.
df['일시'] = pd.to_datetime(df['일시'])

# 2. '일시'를 기준으로 그룹화한 뒤, 숫자 데이터들만 평균(mean)을 냅니다.
# '지점'이나 '지점명' 같은 문자열/ID 칼럼은 평균을 낼 수 없으므로 제외됩니다.
df_daily_avg = df.groupby('일시').mean(numeric_only=True).reset_index()

df_daily_avg = df_daily_avg.drop(columns=['지점'])

# 3. 결과 확인
print("--- 통합된 데이터 상위 5행 ---")
print(df_daily_avg.head())

print(f"\n통합 전 행 개수: {len(df)}")
print(f"통합 후 행 개수: {len(df_daily_avg)}")

--- 통합된 데이터 상위 5행 ---
          일시  평균기온(°C)
0 2020-01-01     5.375
1 2020-01-02     7.950
2 2020-01-03     8.825
3 2020-01-04     8.275
4 2020-01-05     8.825

통합 전 행 개수: 9270
통합 후 행 개수: 2318


In [ ]:
# 1. '일시'에서 월(month)과 일(day) 정보를 추출합니다.
df_daily_avg['월'] = df_daily_avg['일시'].dt.month
df_daily_avg['일'] = df_daily_avg['일시'].dt.day

# 2. 월, 일을 기준으로 그룹화하여 평균기온의 평균을 계산합니다.
# '평균기온(°C)' 칼럼명은 실제 데이터프레임의 칼럼명으로 맞춰주세요.
final_climate_data = df_daily_avg.groupby(['월', '일'])['평균기온(°C)'].mean().reset_index()

# 3. 결과 확인 (1월 1일부터 12월 31일까지 총 366일 데이터)
print("--- 10개년 월-일별 평균 기온 ---")
display(final_climate_data.head())
print(f"\n최종 행 개수: {len(final_climate_data)} (366일에 가까울수록 정상)")

--- 10개년 월-일별 평균 기온 ---


,월,일,평균기온(°C)
0,1,1,5.989286
1,1,2,6.675000
2,1,3,6.510714
3,1,4,6.989286
4,1,5,7.828571



최종 행 개수: 366 (366일에 가까울수록 정상)


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_20_26.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
final_climate_data.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_20_26.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_20_26.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


## 합치는거

In [ ]:
import pandas as pd

# 1. 합칠 파일 리스트 정의 (이미지 순서대로)
file_list = [
    '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_00_09.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_10_19.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/평균기온_20_26.csv'
]

# 2. 각 파일을 읽어서 리스트에 저장
temp_list = []
for file in file_list:
    df = pd.read_csv(file, encoding='cp949') # 한글 깨짐 방지를 위해 cp949 사용
    temp_list.append(df)

# 3. 데이터프레임 합치기 (위아래로 연결)
combined_df = pd.concat(temp_list, ignore_index=True)
display(combined_df.head())
print(combined_df.shape)

,월,일,평균기온(°C)
0,1,1,6.7700
1,1,2,7.1975
2,1,3,7.0625
3,1,4,6.5325
4,1,5,6.5400


(1098, 3)


In [ ]:
# 1. 월(month)과 일(day)을 기준으로 그룹화하여 평균기온의 평균 계산
# .as_index=False를 쓰면 '월', '일'이 컬럼으로 유지되어 보기 편합니다.
daily_avg = combined_df.groupby(['월', '일'], as_index=False)['평균기온(°C)'].mean()

# 2. 결과 출력 (상위 5개 데이터)
display(daily_avg.head())

# 3. 전체 데이터 개수 확인 (윤달 포함 시 366개 내외)
print(daily_avg.shape)

,월,일,평균기온(°C)
0,1,1,6.281429
1,1,2,7.050833
2,1,3,6.694405
3,1,4,6.760595
4,1,5,6.962024


(366, 3)


In [ ]:
import pandas as pd

# 1. 2024-06-01부터 2026-05-09까지 1시간 간격의 시간 범위 생성
start_date = "2024-06-01 00:00:00"
end_date = "2026-05-09 23:00:00"
date_range = pd.date_range(start=start_date, end=end_date, freq='H')

# 2. 결과 데이터프레임 생성
new_df = pd.DataFrame({'Timestamp': date_range})

# 3. '월'과 '일' 추출 (매핑을 위해)
new_df['월'] = new_df['Timestamp'].dt.month
new_df['일'] = new_df['Timestamp'].dt.day

# 4. 기존 daily_avg 데이터와 병합 (Merge)
# '월'과 '일'을 기준으로 매칭하여 평균기온을 가져옵니다.
result_df = pd.merge(new_df, daily_avg, on=['월', '일'], how='left')

# 5. 요청하신 포맷(날짜, 시간 따로 분리)으로 정리
result_df['날짜'] = result_df['Timestamp'].dt.strftime('%Y-%m-%d')
result_df['시간'] = result_df['Timestamp'].dt.strftime('%H:%M:%S')

# 6. 최종 컬럼 순서 및 선택
final_result = result_df[['날짜', '시간', '평균기온(°C)']]

# 결과 확인
display(final_result.head(30))

,날짜,시간,평균기온(°C)
0,2024-06-01,00:00:00,19.764722
1,2024-06-01,01:00:00,19.764722
2,2024-06-01,02:00:00,19.764722
3,2024-06-01,03:00:00,19.764722
4,2024-06-01,04:00:00,19.764722
5,2024-06-01,05:00:00,19.764722
6,2024-06-01,06:00:00,19.764722
7,2024-06-01,07:00:00,19.764722
8,2024-06-01,08:00:00,19.764722
9,2024-06-01,09:00:00,19.764722


In [ ]:
missing_counts = final_result.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
날짜          0
시간          0
평균기온(°C)    0
dtype: int64


In [ ]:
import pandas as pd

# 1. 기상데이터.csv 로드 (인코딩 옵션 추가)
weather_data_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/기상데이터.csv'

try:
    # 한국어 환경에서 가장 흔한 인코딩인 cp949 시도
    weather_df = pd.read_csv(weather_data_path, encoding='cp949')
except UnicodeDecodeError:
    # 안 될 경우 euc-kr 시도
    weather_df = pd.read_csv(weather_data_path, encoding='euc-kr')

# 2. 이후 병합 과정은 동일합니다.
weather_df['날짜'] = pd.to_datetime(weather_df['날짜']).dt.strftime('%Y-%m-%d')
weather_df['시간'] = pd.to_datetime(weather_df['시간'], format='%H:%M:%S').dt.strftime('%H:%M:%S')

weather_temp = weather_df[['날짜', '시간', '기온(°C)']]

# 평년 데이터와 실제 기온 데이터 병합
total_result = pd.merge(final_result, weather_temp, on=['날짜', '시간'], how='left')

# 컬럼명 정리
total_result = total_result.rename(columns={
    '평균기온(°C)': '평년기온(°C)',
    '기온(°C)': '기온(°C)'
})

print("병합 완료!")
display(total_result.tail(10))

병합 완료!


,날짜,시간,평년기온(°C),기온(°C)
16982,2026-05-09,14:00:00,17.2625,NaN
16983,2026-05-09,15:00:00,17.2625,NaN
16984,2026-05-09,16:00:00,17.2625,NaN
16985,2026-05-09,17:00:00,17.2625,NaN
16986,2026-05-09,18:00:00,17.2625,NaN
16987,2026-05-09,19:00:00,17.2625,NaN
16988,2026-05-09,20:00:00,17.2625,NaN
16989,2026-05-09,21:00:00,17.2625,NaN
16990,2026-05-09,22:00:00,17.2625,NaN
16991,2026-05-09,23:00:00,17.2625,NaN


In [ ]:
print(total_result.shape)

(16992, 4)


In [ ]:
# 1. 결측치가 있는 행 제거 (실제기온(°C) 등이 NaN인 경우 삭제)
# subset을 지정하지 않으면 어느 한 컬럼이라도 NaN인 행을 모두 지웁니다.
total_result_cleaned = total_result.dropna()

# 2. 인덱스 재정렬 (행을 지우면 번호가 건너뛰어지므로 다시 매김)
total_result_cleaned = total_result_cleaned.reset_index(drop=True)

print(f"제거 전 행 개수: {len(total_result)}")
print(f"제거 후 행 개수: {len(total_result_cleaned)}")

# 결과 확인
display(total_result_cleaned.tail())

제거 전 행 개수: 16992
제거 후 행 개수: 16930


,날짜,시간,평년기온(°C),기온(°C)
16925,2026-05-07,05:00:00,17.278056,14.950
16926,2026-05-07,06:00:00,17.278056,14.800
16927,2026-05-07,07:00:00,17.278056,16.225
16928,2026-05-07,08:00:00,17.278056,18.725
16929,2026-05-07,09:00:00,17.278056,20.200


In [ ]:
# 1. 날짜 데이터 타입으로 일시적 변환 (필터링 및 그룹화를 위해)
total_result['날짜_dt'] = pd.to_datetime(total_result['날짜'])

# 2. 2026년 5월 6일까지만 필터링
final_filtered = total_result[total_result['날짜_dt'] <= '2026-05-06'].copy()

# 3. 일평균 기온 계산
# '날짜'를 기준으로 그룹화한 뒤 '기온(°C)'의 평균을 구해 각 행에 할당합니다.
final_filtered['일평균기온(°C)'] = final_filtered.groupby('날짜')['기온(°C)'].transform('mean')

# 4. 불필요해진 임시 컬럼 삭제 및 정리
final_filtered = final_filtered.drop(columns=['날짜_dt'])

# 5. 결과 확인
print("2026-05-06까지 필터링 및 일평균 계산 완료!")
display(final_filtered.tail(30)) # 마지막 부분(5월 6일 포함) 확인

2026-05-06까지 필터링 및 일평균 계산 완료!


,날짜,시간,평년기온(°C),기온(°C),일평균기온(°C)
16890,2026-05-05,18:00:00,17.570000,18.000,15.078125
16891,2026-05-05,19:00:00,17.570000,16.475,15.078125
16892,2026-05-05,20:00:00,17.570000,15.250,15.078125
16893,2026-05-05,21:00:00,17.570000,14.425,15.078125
16894,2026-05-05,22:00:00,17.570000,13.900,15.078125
16895,2026-05-05,23:00:00,17.570000,13.500,15.078125
16896,2026-05-06,00:00:00,17.180357,13.275,17.013542
16897,2026-05-06,01:00:00,17.180357,13.000,17.013542
16898,2026-05-06,02:00:00,17.180357,12.900,17.013542
16899,2026-05-06,03:00:00,17.180357,12.975,17.013542


In [ ]:
final_filtered.head()

,날짜,시간,평년기온(°C),기온(°C),일평균기온(°C)
0,2024-06-01,00:00:00,19.764722,17.750,19.605208
1,2024-06-01,01:00:00,19.764722,17.775,19.605208
2,2024-06-01,02:00:00,19.764722,17.625,19.605208
3,2024-06-01,03:00:00,19.764722,17.650,19.605208
4,2024-06-01,04:00:00,19.764722,17.600,19.605208


In [ ]:
# 1. 기온편차 계산 (실제 일평균 - 평년기온)
# 실제 측정된 하루 평균값에서 해당 날짜의 평년 기온을 뺍니다.
final_filtered['기온편차(°C)'] = final_filtered['일평균기온(°C)'] - final_filtered['평년기온(°C)']

# 2. 소수점 셋째 자리까지 반올림 (보기 편하게 정리)
final_filtered['기온편차(°C)'] = final_filtered['기온편차(°C)'].round(3)

# 3. 요청하신 순서대로 컬럼 재배치
# 날짜, 시간, 실제기온, 실제_일평균기온, 기온편차, 평년기온 순서
cols = ['날짜', '시간', '기온(°C)', '평년기온(°C)','일평균기온(°C)','기온편차(°C)']
final_result_with_dev = final_filtered[cols]

# 4. 결과 출력
print("일평균기온 옆에 기온편차 컬럼이 추가되었습니다.")
display(final_result_with_dev.head(25))

일평균기온 옆에 기온편차 컬럼이 추가되었습니다.


,날짜,시간,기온(°C),평년기온(°C),일평균기온(°C),기온편차(°C)
0,2024-06-01,00:00:00,17.750,19.764722,19.605208,-0.16
1,2024-06-01,01:00:00,17.775,19.764722,19.605208,-0.16
2,2024-06-01,02:00:00,17.625,19.764722,19.605208,-0.16
3,2024-06-01,03:00:00,17.650,19.764722,19.605208,-0.16
4,2024-06-01,04:00:00,17.600,19.764722,19.605208,-0.16
5,2024-06-01,05:00:00,17.600,19.764722,19.605208,-0.16
6,2024-06-01,06:00:00,17.925,19.764722,19.605208,-0.16
7,2024-06-01,07:00:00,18.825,19.764722,19.605208,-0.16
8,2024-06-01,08:00:00,19.775,19.764722,19.605208,-0.16
9,2024-06-01,09:00:00,20.300,19.764722,19.605208,-0.16


In [ ]:
# 1. 필요한 컬럼만 선택하여 데이터프레임 갱신
# '날짜', '시간'은 데이터의 식별을 위해 남겨두고, 결과값인 '기온편차(°C)'만 포함합니다.
final_result_only_dev = final_filtered[['날짜', '시간', '기온편차(°C)']]

# 2. 결과 확인
print("기온편차를 제외한 나머지 중간 계산 컬럼 삭제 완료!")
display(final_result_only_dev.head(25))

기온편차를 제외한 나머지 중간 계산 컬럼 삭제 완료!


,날짜,시간,기온편차(°C)
0,2024-06-01,00:00:00,-0.16
1,2024-06-01,01:00:00,-0.16
2,2024-06-01,02:00:00,-0.16
3,2024-06-01,03:00:00,-0.16
4,2024-06-01,04:00:00,-0.16
5,2024-06-01,05:00:00,-0.16
6,2024-06-01,06:00:00,-0.16
7,2024-06-01,07:00:00,-0.16
8,2024-06-01,08:00:00,-0.16
9,2024-06-01,09:00:00,-0.16


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/기온편차.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
final_result_only_dev.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/기온편차.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/기온편차.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


# 전력 수요

In [ ]:
import pandas as pd

file_list = [
    '/content/drive/MyDrive/ax_team/ax_data/model_a/시간별 제주전력수요_240401_240630.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/시간별 제주전력수요_240701_240930.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/시간별 제주전력수요_241001_241231.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/시간별 제주전력수요_250101_250930.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/시간별 제주전력수요_251001_251231.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/시간별 제주전력수요_260101_260331.csv'
]

temp_list = []

print("--- 각 파일별 데이터 크기(Shape) 확인 ---")
for f in file_list:
    file_name = f.split('/')[-1]
    try:
        # 1. 기본적으로 cp949로 시도
        df = pd.read_csv(f, encoding='cp949')
        print(f"성공(cp949): {file_name:40} | 크기: {df.shape}")
        temp_list.append(df)
    except:
        try:
            # 2. 실패 시 utf-8로 재시도
            df = pd.read_csv(f, encoding='utf-8')
            print(f"성공(utf-8):  {file_name:40} | 크기: {df.shape}")
            temp_list.append(df)
        except Exception as e:
            # 3. 둘 다 안 될 경우 에러 메시지
            print(f"❌ 실패: {file_name:40} | 사유: {e}")

print("-" * 75)

# 2. 데이터 통합
if temp_list:
    combined_power_df = pd.concat(temp_list, ignore_index=True)
    print(f"✅ 모든 파일 통합 완료!")
    print(f"최종 통합 데이터 크기: {combined_power_df.shape}")
    print("-" * 75)

    display(combined_power_df.head())

--- 각 파일별 데이터 크기(Shape) 확인 ---
성공(cp949): 시간별 제주전력수요_240401_240630.csv             | 크기: (91, 25)
성공(utf-8):  시간별 제주전력수요_240701_240930.csv             | 크기: (92, 25)
성공(cp949): 시간별 제주전력수요_241001_241231.csv             | 크기: (92, 25)
성공(cp949): 시간별 제주전력수요_250101_250930.csv             | 크기: (273, 25)
성공(cp949): 시간별 제주전력수요_251001_251231.csv             | 크기: (92, 25)
성공(cp949): 시간별 제주전력수요_260101_260331.csv             | 크기: (90, 25)
---------------------------------------------------------------------------
✅ 모든 파일 통합 완료!
최종 통합 데이터 크기: (730, 25)
---------------------------------------------------------------------------


,날짜,1시,2시,3시,4시,5시,6시,7시,8시,9시,...,15시,16시,17시,18시,19시,20시,21시,22시,23시,24시
0,2024-04-01,720.711,690.301,670.386,663.605,667.734,682.269,708.508,667.352,617.870,...,563.236,578.780,617.004,667.8,728.341,805.237,819.32,811.545,793.193,763.039
1,2024-04-02,709.763,670.539,649.518,627.991,618.201,628.515,656.501,676.913,685.929,...,716.473,740.468,771.365,784.352,787.469,797.451,780.649,751.362,730.794,703.806
2,2024-04-03,650.175,616.703,593.682,578.415,590.680,608.402,632.192,681.993,729.439,...,730.823,710.308,713.965,731.292,752.797,773.795,765.806,750.222,736.659,714.506
3,2024-04-04,669.152,637.872,617.979,604.064,604.189,622.193,651.588,685.232,698.120,...,721.947,724.128,747.563,785.912,804.015,817.774,802.169,775.770,754.872,734.167
4,2024-04-05,686.584,658.057,649.629,638.794,634.174,655.346,688.157,679.687,656.453,...,604.534,609.486,637.197,672.954,732.188,811.615,823.787,813.885,799.799,773.808


In [ ]:
missing_counts = combined_power_df.isnull().sum()

print("--- 각 칼럼별 결측치 개수 ---")
print(missing_counts)

--- 각 칼럼별 결측치 개수 ---
날짜     0
1시     0
2시     0
3시     0
4시     0
5시     0
6시     0
7시     0
8시     0
9시     0
10시    0
11시    0
12시    0
13시    0
14시    0
15시    0
16시    0
17시    0
18시    0
19시    0
20시    0
21시    0
22시    0
23시    0
24시    0
dtype: int64


In [ ]:
import pandas as pd

# 1. 가로로 나열된 시간 컬럼들을 세로로 녹이기 (Melt)
# '날짜' 컬럼을 기준으로 나머지 1시~24시 컬럼을 '시간'과 '전력수요'로 변환합니다.
long_df = combined_power_df.melt(id_vars=['날짜'], var_name='시간', value_name='전력수요')

# 2. 시간 데이터 정리 (숫자만 추출)
# '1시', '2시' 등에서 숫자만 남기고 정수형으로 변환합니다.
long_df['시간'] = long_df['시간'].str.replace('시', '').astype(int)

# 3. 날짜와 시간 컬럼을 합쳐서 실제 시계열 데이터(datetime) 만들기
# 24시는 pd.to_datetime이 인식하지 못하므로, 임시로 처리 후 나중에 조정합니다.
long_df['일시'] = pd.to_datetime(long_df['날짜'])

# 4. 24시 처리 로직: 24시는 다음날 0시로 변경
# '시간'이 24인 행을 찾아 날짜를 하루 더하고 시간을 0으로 바꿉니다.
mask_24 = long_df['시간'] == 24
long_df.loc[mask_24, '일시'] += pd.Timedelta(days=1)
long_df.loc[mask_24, '시간'] = 0

# 5. 첫 날 0시 데이터 추가 (2024-04-01 0시)
# 요청하신 대로 2024-04-01 1시 데이터를 복사해서 사용합니다.
first_day_1hr = long_df[(long_df['날짜'] == '2024-04-01') & (long_df['시간'] == 1)].copy()
first_day_1hr['시간'] = 0
first_day_1hr['일시'] = pd.to_datetime('2024-04-01 00:00:00')

# 기존 데이터에 추가
final_df = pd.concat([first_day_1hr, long_df], ignore_index=True)

# 6. 정렬 및 불필요한 컬럼 정리
# 시간 순서대로 정렬하고, 기존의 '날짜' 컬럼 대신 새로 만든 '일시'와 '시간'을 사용합니다.
final_df = final_df.sort_values(by='일시').reset_index(drop=True)

# 결과물에서 '날짜'는 삭제하고 '일시'를 '날짜' 형식으로 재구성 (필요시)
final_df = final_df[['일시', '시간', '전력수요']]
final_df.columns = ['날짜', '시간', '전력수요']

# 7. 결과 확인
display(final_df.head(10))
print(f"최종 데이터 크기: {final_df.shape}")

,날짜,시간,전력수요
0,2024-04-01,0,720.711
1,2024-04-01,21,819.32
2,2024-04-01,2,690.301
3,2024-04-01,20,805.237
4,2024-04-01,19,728.341
5,2024-04-01,18,667.8
6,2024-04-01,17,617.004
7,2024-04-01,16,578.78
8,2024-04-01,3,670.386
9,2024-04-01,15,563.236


최종 데이터 크기: (17521, 3)


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 컬럼별 결측치 합계 ---")
print(final_df.isnull().sum())

# 2. 결측치가 하나라도 있는 행이 있는지 확인 (True/False)
has_nan = final_df.isnull().values.any()
print(f"\n데이터프레임 내 결측치 존재 여부: {has_nan}")

# 3. (옵션) 결측치가 포함된 행만 직접 확인하고 싶을 때
nan_rows = final_df[final_df.isnull().any(axis=1)]
print("\n--- 결측치가 포함된 행 (상위 5개) ---")
print(nan_rows.head())

--- 컬럼별 결측치 합계 ---
날짜      0
시간      0
전력수요    0
dtype: int64

데이터프레임 내 결측치 존재 여부: False

--- 결측치가 포함된 행 (상위 5개) ---
Empty DataFrame
Columns: [날짜, 시간, 전력수요]
Index: []


In [ ]:
import pandas as pd

# 1. '날짜' 컬럼을 확실하게 datetime 형식으로 변환
final_df['날짜'] = pd.to_datetime(final_df['날짜'])

# 2. 시간을 00:00:00 형식의 문자열로 변환
# 숫자를 2자리 문자열로 만들고(zfill), 뒤에 :00:00을 붙입니다.
final_df['시간'] = final_df['시간'].astype(int).apply(lambda x: f"{str(x).zfill(2)}:00:00")

# 3. 날짜와 시간을 기준으로 오름차순 정렬
# 날짜 순으로 먼저 세우고, 같은 날짜 안에서는 시간 순으로 정렬합니다.
final_df = final_df.sort_values(by=['날짜', '시간']).reset_index(drop=True)

# 4. 결과 확인
display(final_df.head(25)) # 4월 1일 0시부터 24시간치 확인
print(f"데이터 타입 확인:\n{final_df.dtypes}")
print(final_df.shape)

,날짜,시간,전력수요
0,2024-04-01,00:00:00,720.711
1,2024-04-01,01:00:00,720.711
2,2024-04-01,02:00:00,690.301
3,2024-04-01,03:00:00,670.386
4,2024-04-01,04:00:00,663.605
5,2024-04-01,05:00:00,667.734
6,2024-04-01,06:00:00,682.269
7,2024-04-01,07:00:00,708.508
8,2024-04-01,08:00:00,667.352
9,2024-04-01,09:00:00,617.87


데이터 타입 확인:
날짜      datetime64[ns]
시간              object
전력수요            object
dtype: object
(17521, 3)


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 컬럼별 결측치 합계 ---")
print(final_df.isnull().sum())

# 2. 결측치가 하나라도 있는 행이 있는지 확인 (True/False)
has_nan = final_df.isnull().values.any()
print(f"\n데이터프레임 내 결측치 존재 여부: {has_nan}")

# 3. (옵션) 결측치가 포함된 행만 직접 확인하고 싶을 때
nan_rows = final_df[final_df.isnull().any(axis=1)]
print("\n--- 결측치가 포함된 행 (상위 5개) ---")
print(nan_rows.head())

--- 컬럼별 결측치 합계 ---
날짜      0
시간      0
전력수요    0
dtype: int64

데이터프레임 내 결측치 존재 여부: False

--- 결측치가 포함된 행 (상위 5개) ---
Empty DataFrame
Columns: [날짜, 시간, 전력수요]
Index: []


In [ ]:
import pandas as pd

# 1. 숫자 변환 (변환 중 생긴 NaN은 일단 둡니다)
final_df['전력수요'] = pd.to_numeric(final_df['전력수요'], errors='coerce')

# 2. [해결책] 원본 전력수요에 생긴 미세한 구멍(20개)을 먼저 채웁니다.
# 바로 직전의 정상적인 값으로 채우는 방식입니다.
final_df['전력수요'] = final_df['전력수요'].ffill()

# 3. 칼럼 생성 (원본이 깨끗해졌으므로 결과도 깨끗해집니다)
final_df['1시간전_수요'] = final_df['전력수요'].shift(1)
final_df['전일_동시간_수요'] = final_df['전력수요'].shift(24)

# 4. 최근 7일 평균 (min_periods=1 설정)
# 168개가 다 안 모였어도 현재까지 모인 데이터만으로 평균을 내서 NaN을 방지합니다.
final_df['최근7일_평균_수요'] = final_df['전력수요'].shift(1).rolling(window=168, min_periods=1).mean()

# 5. 6월 1일부터 자르기 (앞부분의 구조적 결측치들을 제거하는 효과)
final_df = final_df[final_df['날짜'] >= '2024-06-01'].copy()

# 결과 확인
print(final_df.isnull().sum())
display(final_df.head())


날짜            0
시간            0
전력수요          0
1시간전_수요       0
전일_동시간_수요     0
최근7일_평균_수요    0
dtype: int64


,날짜,시간,전력수요,1시간전_수요,전일_동시간_수요,최근7일_평균_수요
1464,2024-06-01,00:00:00,654.877,684.304,648.583,624.865643
1465,2024-06-01,01:00:00,615.135,654.877,602.835,624.857607
1466,2024-06-01,02:00:00,580.990,615.135,569.661,624.875625
1467,2024-06-01,03:00:00,562.764,580.990,551.240,624.903429
1468,2024-06-01,04:00:00,547.270,562.764,538.764,624.955512


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 컬럼별 결측치 합계 ---")
print(final_df.isnull().sum())

# 2. 결측치가 하나라도 있는 행이 있는지 확인 (True/False)
has_nan = final_df.isnull().values.any()
print(f"\n데이터프레임 내 결측치 존재 여부: {has_nan}")

# 3. (옵션) 결측치가 포함된 행만 직접 확인하고 싶을 때
nan_rows = final_df[final_df.isnull().any(axis=1)]
print("\n--- 결측치가 포함된 행 (상위 5개) ---")
print(nan_rows.head())

--- 컬럼별 결측치 합계 ---
날짜            0
시간            0
전력수요          0
1시간전_수요       0
전일_동시간_수요     0
최근7일_평균_수요    0
dtype: int64

데이터프레임 내 결측치 존재 여부: False

--- 결측치가 포함된 행 (상위 5개) ---
Empty DataFrame
Columns: [날짜, 시간, 전력수요, 1시간전_수요, 전일_동시간_수요, 최근7일_평균_수요]
Index: []


In [ ]:
print(final_df.shape)

(16057, 6)


In [ ]:
# 1. 2024년 6월 1일 이후 데이터만 필터링 (기존 데이터 제거)
final_df = final_df[final_df['날짜'] >= '2024-06-01'].copy()

# 2. 인덱스 재설정 (데이터를 잘라냈으므로 번호를 0부터 다시 매깁니다)
final_df = final_df.reset_index(drop=True)

# 결과 확인
display(final_df.head())
print(final_df.shape)

,날짜,시간,전력수요,1시간전_수요,전일_동시간_수요,최근7일_평균_수요
0,2024-06-01,00:00:00,654.877,684.304,648.583,624.865643
1,2024-06-01,01:00:00,615.135,654.877,602.835,624.857607
2,2024-06-01,02:00:00,580.990,615.135,569.661,624.875625
3,2024-06-01,03:00:00,562.764,580.990,551.240,624.903429
4,2024-06-01,04:00:00,547.270,562.764,538.764,624.955512


(16057, 6)


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 컬럼별 결측치 합계 ---")
print(final_df.isnull().sum())

# 2. 결측치가 하나라도 있는 행이 있는지 확인 (True/False)
has_nan = final_df.isnull().values.any()
print(f"\n데이터프레임 내 결측치 존재 여부: {has_nan}")

# 3. (옵션) 결측치가 포함된 행만 직접 확인하고 싶을 때
nan_rows = final_df[final_df.isnull().any(axis=1)]
print("\n--- 결측치가 포함된 행 (상위 5개) ---")
print(nan_rows.head())

--- 컬럼별 결측치 합계 ---
날짜            0
시간            0
전력수요          0
1시간전_수요       0
전일_동시간_수요     0
최근7일_평균_수요    0
dtype: int64

데이터프레임 내 결측치 존재 여부: False

--- 결측치가 포함된 행 (상위 5개) ---
Empty DataFrame
Columns: [날짜, 시간, 전력수요, 1시간전_수요, 전일_동시간_수요, 최근7일_평균_수요]
Index: []


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/전체전력수요.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
final_df.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/전체전력수요.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/전체전력수요.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


# 관광객


In [ ]:
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/제주 일일 관광객 수.csv'

try:
    # 1. 일반적인 utf-8 시도
    df_tourist = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    # 2. 엑셀 저장용 utf-8-sig 시도
    df_tourist = pd.read_csv(file_path, encoding='utf-8-sig')

# 이후 날짜 전처리 코드
df_tourist['기준일자(YYYYMMDD)'] = pd.to_datetime(df_tourist['기준일자(YYYYMMDD)'], format='%Y%m%d')
df_tourist = df_tourist.rename(columns={'기준일자(YYYYMMDD)': '날짜'})
df_tourist = df_tourist.drop('전체방문객수', axis=1)

display(df_tourist.head())

,날짜,내국인방문객수,외국인방문객수
0,2026-05-06,28842,5271
1,2026-05-05,24868,6337
2,2026-05-04,26032,9098
3,2026-05-03,31428,14406
4,2026-05-02,41052,5712


In [ ]:
df_tourist = df_tourist.sort_values(by='날짜', ascending=True).reset_index(drop=True)

# 결과 확인
display(df_tourist.head())

,날짜,내국인방문객수,외국인방문객수
0,2024-06-01,37381,5735
1,2024-06-02,34581,5946
2,2024-06-03,37368,3245
3,2024-06-04,35583,8501
4,2024-06-05,39653,5920


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 컬럼별 결측치 합계 ---")
print(df_tourist.isnull().sum())

# 2. 결측치가 하나라도 있는 행이 있는지 확인 (True/False)
has_nan = df_tourist.isnull().values.any()
print(f"\n데이터프레임 내 결측치 존재 여부: {has_nan}")

# 3. (옵션) 결측치가 포함된 행만 직접 확인하고 싶을 때
nan_rows = df_tourist[df_tourist.isnull().any(axis=1)]
print("\n--- 결측치가 포함된 행 (상위 5개) ---")
print(nan_rows.head())

--- 컬럼별 결측치 합계 ---
날짜         0
내국인방문객수    0
외국인방문객수    0
dtype: int64

데이터프레임 내 결측치 존재 여부: False

--- 결측치가 포함된 행 (상위 5개) ---
Empty DataFrame
Columns: [날짜, 내국인방문객수, 외국인방문객수]
Index: []


In [ ]:
import pandas as pd

# 1. 0시부터 23시까지의 시간 리스트 생성 (00:00:00 포맷)
# 0부터 23까지 숫자를 가져와서 'HH:MM:SS' 문자열로 변환합니다.
hours_list = [f"{h:02d}:00:00" for h in range(24)]
df_hours = pd.DataFrame({'시간': hours_list})

# 2. 기존 df_tourist와 시간 데이터프레임을 cross 조인하여 확장
# 모든 날짜에 대해 24개의 시간 행이 생성됩니다.
df_expanded = df_tourist.merge(df_hours, how='cross')

# 3. 칼럼 순서 재배치 (날짜 바로 옆에 시간이 오도록)
cols = ['날짜', '시간', '내국인방문객수', '외국인방문객수']
df_expanded = df_expanded[cols]

# 결과 확인
display(df_expanded.head(30))

,날짜,시간,내국인방문객수,외국인방문객수
0,2024-06-01,00:00:00,37381,5735
1,2024-06-01,01:00:00,37381,5735
2,2024-06-01,02:00:00,37381,5735
3,2024-06-01,03:00:00,37381,5735
4,2024-06-01,04:00:00,37381,5735
5,2024-06-01,05:00:00,37381,5735
6,2024-06-01,06:00:00,37381,5735
7,2024-06-01,07:00:00,37381,5735
8,2024-06-01,08:00:00,37381,5735
9,2024-06-01,09:00:00,37381,5735


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/관광객입도수.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
df_expanded.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/관광객입도수.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/관광객입도수.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


# 생산지수

In [ ]:
import pandas as pd

# 1. 파일 불러오기
file_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/제주 생산지수.csv'
df = pd.read_csv(file_path, encoding='cp949')

# 2. 필요한 행만 필터링 (총지수, 광업 및 제조업, 전기업 및 가스업)
# '월별' 컬럼의 값들 중 우리가 필요한 것만 골라냅니다.
target_rows = ['총지수', '광업 및 제조업', '전기업 및 가스업']
df_filtered = df[df['월별'].isin(target_rows)].copy()

# 3. 행과 열 뒤집기 (Transpose)
# '월별'을 인덱스로 잡고 뒤집어서 날짜가 행으로 오게 만듭니다.
df_t = df_filtered.set_index('월별').T.reset_index()

# 4. 컬럼명 정리
df_t.columns.name = None
df_t = df_t.rename(columns={
    'index': '날짜',
    '총지수': '총 생산지수',
    '광업 및 제조업': '광업 및 제조업 생산지수',
    '전기업 및 가스업': '전기업 및 가스업 생산지수'
})

# 5. 날짜 형식 변경 (2024.06 -> 2024-06)
# 잠정치 표시(p) 제거 및 공백 정리
df_t['날짜'] = df_t['날짜'].str.replace(r' p\)', '', regex=True).str.strip()

# '2024.1' -> '2024.01' 보정 함수
def fix_date(date_str):
    if '.' in date_str:
        year, month = date_str.split('.')
        return f"{year}-{month.zfill(2)}"
    return date_str

df_t['날짜'] = df_t['날짜'].apply(fix_date)

# 6. 데이터 타입 정리 (지수 값들을 숫자형으로 변환)
cols_to_fix = ['총 생산지수', '광업 및 제조업 생산지수', '전기업 및 가스업 생산지수']
for col in cols_to_fix:
    if col in df_t.columns:
        df_t[col] = pd.to_numeric(df_t[col], errors='coerce')

# 7. 최종 결과 확인
print("--- 정리된 데이터 확인 ---")
display(df_t.head())

--- 정리된 데이터 확인 ---


,날짜,총 생산지수,광업 및 제조업 생산지수,전기업 및 가스업 생산지수
0,2024-06,107.2,108.4,104.6
1,2024-07,115.0,104.3,140.1
2,2024-08,124.0,112.5,151.1
3,2024-09,102.5,92.1,126.9
4,2024-01,108.0,105.3,114.2


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 컬럼별 결측치 합계 ---")
print(df_t.isnull().sum())

# 2. 결측치가 하나라도 있는 행이 있는지 확인 (True/False)
has_nan = df_t.isnull().values.any()
print(f"\n데이터프레임 내 결측치 존재 여부: {has_nan}")

# 3. (옵션) 결측치가 포함된 행만 직접 확인하고 싶을 때
nan_rows = df_t[df_t.isnull().any(axis=1)]
print("\n--- 결측치가 포함된 행 (상위 5개) ---")
print(nan_rows.head())

--- 컬럼별 결측치 합계 ---
날짜                0
총 생산지수            0
광업 및 제조업 생산지수     0
전기업 및 가스업 생산지수    0
dtype: int64

데이터프레임 내 결측치 존재 여부: False

--- 결측치가 포함된 행 (상위 5개) ---
Empty DataFrame
Columns: [날짜, 총 생산지수, 광업 및 제조업 생산지수, 전기업 및 가스업 생산지수]
Index: []


In [ ]:
import pandas as pd

# 1. '날짜'가 인덱스에 있다면 칼럼으로 복구
if '날짜' not in df_t.columns:
    df_t = df_t.reset_index()

# 2. '날짜' 칼럼 변환 및 중복 제거
df_t['날짜'] = pd.to_datetime(df_t['날짜'])
# 같은 달이 여러 개 있으면 '첫 번째'로 등장한 행만 남깁니다.
df_t = df_t.drop_duplicates(subset=['날짜'], keep='first')

# 3. 인덱스 설정 (정렬하지 않음 - 데이터 순서 유지)
df_t = df_t.set_index('날짜')

# 4. 사용자가 의도한 시작(2024-06)부터 끝(2026-03 말일)까지의 범위 생성
start_date = pd.to_datetime('2024-06-01')
end_date = pd.to_datetime('2026-03-01') + pd.offsets.MonthEnd(0)

# 5. 일 단위 확장 및 값 채우기
new_range_daily = pd.date_range(start_date, end_date, freq='D')
df_daily = df_t.reindex(new_range_daily).ffill()

# 6. 시간 단위 확장 (00:00:00 ~ 23:00:00)
hourly_range = pd.date_range(start_date, end_date + pd.Timedelta(hours=23), freq='H')
df_hourly = df_daily.reindex(hourly_range).ffill()

# 7. 최종 포맷팅
df_final = df_hourly.reset_index().rename(columns={'index': 'datetime'})
df_final['날짜'] = df_final['datetime'].dt.strftime('%Y-%m-%d')
df_final['시간'] = df_final['datetime'].dt.strftime('%H:%M:%S')

# 필요한 칼럼만 추출 (날짜, 시간, 생산지수 칼럼들)
target_cols = ['날짜', '시간'] + [c for c in df_final.columns if c not in ['날짜', '시간', 'datetime']]
df_t = df_final[target_cols]

# 2024-06-01 00:00:00부터 시작하는지 확인
display(df_t.head(5))

,날짜,시간,총 생산지수,광업 및 제조업 생산지수,전기업 및 가스업 생산지수
0,2024-06-01,00:00:00,107.2,108.4,104.6
1,2024-06-01,01:00:00,107.2,108.4,104.6
2,2024-06-01,02:00:00,107.2,108.4,104.6
3,2024-06-01,03:00:00,107.2,108.4,104.6
4,2024-06-01,04:00:00,107.2,108.4,104.6


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 컬럼별 결측치 합계 ---")
print(df_t.isnull().sum())

# 2. 결측치가 하나라도 있는 행이 있는지 확인 (True/False)
has_nan = df_t.isnull().values.any()
print(f"\n데이터프레임 내 결측치 존재 여부: {has_nan}")

# 3. (옵션) 결측치가 포함된 행만 직접 확인하고 싶을 때
nan_rows = df_t[df_t.isnull().any(axis=1)]
print("\n--- 결측치가 포함된 행 (상위 5개) ---")
print(nan_rows.head())

--- 컬럼별 결측치 합계 ---
날짜                0
시간                0
총 생산지수            0
광업 및 제조업 생산지수     0
전기업 및 가스업 생산지수    0
dtype: int64

데이터프레임 내 결측치 존재 여부: False

--- 결측치가 포함된 행 (상위 5개) ---
Empty DataFrame
Columns: [날짜, 시간, 총 생산지수, 광업 및 제조업 생산지수, 전기업 및 가스업 생산지수]
Index: []


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/전체생산지수.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
df_t.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/전체생산지수.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/전체생산지수.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!


# 모델 A 학습 데이터

In [ ]:
import pandas as pd

# 1. 병합할 파일 리스트 정의
file_names = [
    '/content/drive/MyDrive/ax_team/ax_data/model_a/전처리/전체생산지수.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/전처리/전체전력수요.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/전처리/day_info.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/전처리/관광객입도수.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/전처리/기상데이터.csv',
    '/content/drive/MyDrive/ax_team/ax_data/model_a/전처리/기온편차.csv'
]

# 2. 첫 번째 파일을 기준으로 데이터프레임 생성
merged_df = pd.read_csv(file_names[0], encoding='cp949')

# 3. 나머지 파일들을 '날짜'와 '시간' 컬럼을 기준으로 순차적 병합
for file in file_names[1:]:
    # 각 파일을 읽을 때도 인코딩 지정
    next_df = pd.read_csv(file, encoding='cp949')
    merged_df = pd.merge(merged_df, next_df, on=['날짜', '시간'], how='inner')

# 4. 결과 확인 및 저장
print(f"병합 완료! 최종 데이터 형태: {merged_df.shape}")
display(merged_df)


병합 완료! 최종 데이터 형태: (16056, 24)


,날짜,시간,총 생산지수,광업 및 제조업 생산지수,전기업 및 가스업 생산지수,전력수요,1시간전_수요,전일_동시간_수요,최근7일_평균_수요,hour_sin,...,휴일_종류,내국인방문객수,외국인방문객수,기온(°C),강수량(mm),풍속(m/s),습도(%),일사(MJ/m2),전운량(10분위),기온편차(°C)
0,2024-06-01,00:00:00,107.2,108.4,104.6,654.877,684.304,648.583,624.865643,0.000000,...,0,37381,5735,17.750,0.0,2.775,80.50,0.00,3,-0.160
1,2024-06-01,01:00:00,107.2,108.4,104.6,615.135,654.877,602.835,624.857607,0.258819,...,0,37381,5735,17.775,0.0,3.675,79.75,0.00,8,-0.160
2,2024-06-01,02:00:00,107.2,108.4,104.6,580.990,615.135,569.661,624.875625,0.500000,...,0,37381,5735,17.625,0.0,2.650,78.50,0.00,6,-0.160
3,2024-06-01,03:00:00,107.2,108.4,104.6,562.764,580.990,551.240,624.903429,0.707107,...,0,37381,5735,17.650,0.0,3.025,78.00,0.00,5,-0.160
4,2024-06-01,04:00:00,107.2,108.4,104.6,547.270,562.764,538.764,624.955512,0.866025,...,0,37381,5735,17.600,0.0,3.175,80.00,0.00,7,-0.160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16051,2026-03-31,19:00:00,97.7,86.4,124.5,749.392,707.413,784.291,664.124262,-0.965926,...,0,31579,7091,13.500,0.0,2.875,76.50,0.03,10,1.436
16052,2026-03-31,20:00:00,97.7,86.4,124.5,781.466,749.392,790.036,663.956387,-0.866025,...,0,31579,7091,13.200,0.0,2.200,78.50,0.00,10,1.436
16053,2026-03-31,21:00:00,97.7,86.4,124.5,775.632,781.466,766.238,663.883458,-0.707107,...,0,31579,7091,13.025,0.0,1.850,78.75,0.00,10,1.436
16054,2026-03-31,22:00:00,97.7,86.4,124.5,756.583,775.632,736.831,663.826006,-0.500000,...,0,31579,7091,13.175,0.0,1.800,78.75,0.00,10,1.436


In [ ]:
print("--- 컬럼별 결측치 합계 ---")
print(merged_df.isnull().sum())

--- 컬럼별 결측치 합계 ---
날짜                0
시간                0
총 생산지수            0
광업 및 제조업 생산지수     0
전기업 및 가스업 생산지수    0
전력수요              0
1시간전_수요           0
전일_동시간_수요         0
최근7일_평균_수요        0
hour_sin          0
hour_cos          0
요일                0
주말_유무             0
휴일_유무             0
휴일_종류             0
내국인방문객수           0
외국인방문객수           0
기온(°C)            0
강수량(mm)           0
풍속(m/s)           0
습도(%)             0
일사(MJ/m2)         0
전운량(10분위)         0
기온편차(°C)          0
dtype: int64


In [ ]:
# 파일명 설정
output_file_name = '/content/drive/MyDrive/ax_team/ax_data/model_a/model_a_data.csv'

# index=False를 설정해야 0, 1, 2... 같은 행 번호가 파일에 포함되지 않아 깔끔합니다.
merged_df.to_csv(output_file_name, index=False, encoding='cp949')

print(f"성공적으로 저장되었습니다: {output_file_name}")

성공적으로 저장되었습니다: /content/drive/MyDrive/ax_team/ax_data/model_a/model_a_data.csv


In [ ]:
import os

check_path = '/content/drive/MyDrive/ax_team/ax_data/model_a/model_a_data.csv'

if os.path.exists(check_path):
    print("✅ 파일이 드라이브에 존재합니다!")
else:
    print("❌ 파일이 없습니다. 경로를 다시 확인하거나 마운트를 확인하세요.")

✅ 파일이 드라이브에 존재합니다!
